# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreeyeshbaral/ShreeyeshAssignment1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 4 — CTR / Engagement Opportunity Scoring** on the full warehouse release.

This notebook does four things, in order:
1. States the data contract in plain words (5 answers)
2. Classifies every field as feature / label / context / excluded
3. Verifies the contract with three queries, builds five features, and runs the deliberate-leakage trap
4. Names one limitation of this data slice

All queries run on `month=2026-03` (a mid-panel month). The final month (June 2026) is sealed as a test window — never used for label development.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `writing-data-contracts` + `flyrank/flyrank-data`.

In [2]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

Verified data contract query.


In [3]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt
# never fires — if Colab reconnects while a getpass prompt is open, the kernel
# waits on it forever ('Resuming execution...') and you have to restart.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Verified data contract query.


In [4]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients':   f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':   f"read_parquet('{REL}/dim_content.parquet')",
    'fact_march':    f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_sample':   f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}

# Quick connection check — row counts from Parquet metadata (seconds, not minutes)
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

Verified data contract query.


## 1. Unit of analysis + time window

### The contract — five plain-words answers

**1. One row means:** One day of search + analytics performance for one content page at one client. The grain is `report_date × client_hash_id × content_hash_id`.

**2. Table(s) I use:** `fact_content_daily_performance` (partitioned by month) is the main table — daily GSC impressions, clicks, position, plus GA4 sessions and engagement. I join `dim_content` when I need content metadata (type, word count) and `dim_clients` when I need per-client history coverage.

**3. Time window:** I develop on the `month=2026-03` partition (March 2026), a mid-panel month. The full panel spans 2025-01-27 → 2026-06-30 (~17 months). The final month (June 2026) is sealed as a test window — the `_sample` table IS that final month, so I never use it for label logic.

**4. What I predict (label / proxy):** A CTR opportunity score — how far a page's observed click-through rate falls below the expected CTR for its position tier, weighted by impression volume. The proxy label is `is_under_ctr`: 1 when the page's monthly CTR falls below its position tier's median CTR. This is a current-window proxy, not a future outcome.

**5. One thing I deliberately exclude:** `ctr` itself (= clicks / impressions) — it IS the quantity the label is derived from, so including it as a feature would be circular. Also excluded: `gsc_clicks` (direct input to CTR), `trend_direction` and `trend_pct` (label sources for the decline task), and any product-decision flags.

In [6]:
# Discover the schema of our main table so we can verify column names
schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_march']} LIMIT 0").df()
print('=== fact_content_daily_performance (March 2026) columns ===')
for _, row in schema.iterrows():
    print(f"  {row['column_name']:40} {row['column_type']}")

print()
schema_dim = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 0").df()
print('=== dim_content columns ===')
for _, row in schema_dim.iterrows():
    print(f"  {row['column_name']:40} {row['column_type']}")

Verified data contract query.


## 2. Fields: feature / label / context / excluded

Every field I touch goes in exactly one bucket.

### Context (grouping / joining / splitting — never model inputs)

| Field | Why it's context |
|---|---|
| `content_hash_id` | Pseudonymous page ID — for joins and grouping only |
| `client_hash_id` | Pseudonymous client ID — for client-holdout splits |
| `report_date` | Defines the time window; not a model input |
| `ga4_data_available` | Filter flag to exclude zero-filled GA4 rows; not a signal |
| `gsc_data_available` | Filter flag for GSC data presence |
| `position_tier` (computed) | Used for tier-median label computation, then dropped |

### Features (knowable BEFORE the decision moment)

| Feature | Source | Safe because… |
|---|---|---|
| `avg_position_30d` | `AVG(gsc_avg_position)` | Position is measured by GSC daily, before any editorial action |
| `impressions_30d` | `SUM(gsc_impressions)` | Impressions are passively observed search volume |
| `days_with_impressions` | Count of days with > 0 impressions | Counts past days only |
| `engagement_rate_30d` | GA4 engaged sessions / sessions | GA4 engagement is measured independently of search clicks |
| `position_stddev_30d` | `STDDEV(gsc_avg_position)` | Position volatility is observable from past data |

### Label / proxy (the thing we predict — NEVER a feature)

| Field | Role |
|---|---|
| `ctr_30d` (computed) | The rate we score: clicks ÷ impressions × 100. Used to compute the label, then excluded from features |
| `is_under_ctr` (computed) | Binary proxy: 1 when page CTR < its position tier's median CTR |

### Excluded (with a one-line why)

| Field | Why excluded |
|---|---|
| `gsc_clicks` / `ctr` | Direct input to the label computation (CTR = clicks / impressions) — using it is circular |
| `trend_direction` | Label source for the decline task (notebook 02 leakage lesson) |
| `trend_pct` | Computed from `trend_direction`; same leakage risk |
| `_sample` table | It IS the final month (June 2026) = our sealed test window |
| Product flags (`health_score`, etc.) | Encode a decision already made — circular if used as features |

In [8]:
# Show a few sample rows from the March partition to confirm field types
sample_rows = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_avg_position,
           ga4_sessions, ga4_engaged_sessions,
           ga4_data_available, gsc_data_available
    FROM {TABLES['fact_march']}
    WHERE gsc_impressions > 0
    LIMIT 5
""").df()
print('Sample rows from fact_content_daily_performance (March 2026):')
print(sample_rows.to_string(index=False))

Verified data contract query.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

All three verification queries run on `month=2026-03`.

---

### 3.1 — Grain check: one row = one day × one client × one content item

In [10]:
# QUERY 1: Grain check
# If the grain is (report_date, client_hash_id, content_hash_id), then
# GROUP BY those columns should produce no group with count > 1.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print(f'Duplicate grain rows found: {len(grain_check)}')
if len(grain_check) == 0:
    print('✓ Grain confirmed: one row = one (report_date × client × content).')
else:
    print('⚠ Duplicates detected — investigate before modeling.')
    print(grain_check)

Verified data contract query.


### 3.2 — Row count and date span

In [12]:
# QUERY 2: Row count + date span for March 2026
counts = con.sql(f"""
    SELECT
        COUNT(*)                              AS total_rows,
        MIN(report_date)                      AS earliest_date,
        MAX(report_date)                      AS latest_date,
        COUNT(DISTINCT client_hash_id)        AS n_clients,
        COUNT(DISTINCT content_hash_id)       AS n_content_items
    FROM {TABLES['fact_march']}
""").df()

print('March 2026 partition summary:')
print(f"  Total rows:            {counts['total_rows'].iloc[0]:>12,}")
print(f"  Earliest date:         {counts['earliest_date'].iloc[0]}")
print(f"  Latest date:           {counts['latest_date'].iloc[0]}")
print(f"  Distinct clients:      {counts['n_clients'].iloc[0]:>12,}")
print(f"  Distinct content items:{counts['n_content_items'].iloc[0]:>12,}")

Verified data contract query.


### 3.3 — Availability: the `IS TRUE` filter

The `ga4_data_available` flag is **three-valued**: `TRUE`, `FALSE`, or `NULL`. Rows before a client's GA4 start have GA4 columns zero-filled with the flag `FALSE`, but millions of other rows carry `NULL`. Using `= TRUE` or `NOT FALSE` silently mishandles `NULL` rows. The correct filter is `IS TRUE`.

In [14]:
# QUERY 3: Availability — filter with IS TRUE
avail = con.sql(f"""
    SELECT
        COUNT(*)                                                   AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)         AS ga4_available,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE)     AS ga4_not_available,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL)         AS ga4_null,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)         AS gsc_available,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                           AND ga4_data_available IS TRUE)         AS both_available
    FROM {TABLES['fact_march']}
""").df()

total = avail['total_rows'].iloc[0]
ga4_ok = avail['ga4_available'].iloc[0]
ga4_null = avail['ga4_null'].iloc[0]
both_ok = avail['both_available'].iloc[0]

print('Availability breakdown (March 2026):')
print(f"  Total rows:                 {total:>10,}")
print(f"  GA4 available (IS TRUE):    {ga4_ok:>10,}  ({ga4_ok/total:.1%})")
print(f"  GA4 NULL (not TRUE or FALSE):{ga4_null:>9,}  ({ga4_null/total:.1%})")
print(f"  GSC available (IS TRUE):    {avail['gsc_available'].iloc[0]:>10,}")
print(f"  Both available:             {both_ok:>10,}  ({both_ok/total:.1%})")
print()
print('→ The IS TRUE filter is essential: NULL rows are neither TRUE nor FALSE.')
print('  Without it, engagement features would silently include zero-filled rows.')

Verified data contract query.


---

### 3.4 — Five features: build the feature frame

Aggregate the March 2026 daily data to **one row per content item**, then compute the proxy label. Only pages with ≥ 100 impressions and a valid position are kept (the Lane 4 working slice).

In [16]:
# Build the feature frame from month=2026-03
df = con.sql(f"""
    WITH agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            -- Feature 1: average search position
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)
                AS avg_position_30d,
            -- Feature 2: total impressions (volume signal)
            SUM(gsc_impressions) AS impressions_30d,
            -- Feature 3: days with visibility
            COUNT(*) FILTER (WHERE gsc_impressions > 0)
                AS days_with_impressions,
            -- Feature 4: GA4 engagement rate (only where available)
            SUM(ga4_engaged_sessions) FILTER (WHERE ga4_data_available IS TRUE) * 1.0 /
                NULLIF(SUM(ga4_sessions) FILTER (WHERE ga4_data_available IS TRUE), 0)
                AS engagement_rate_30d,
            -- Feature 5: position volatility
            STDDEV_POP(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)
                AS position_stddev_30d,
            -- For label computation only (NOT a feature)
            SUM(gsc_clicks) AS clicks_30d
        FROM {TABLES['fact_march']}
        GROUP BY 1, 2
    )
    SELECT * FROM agg
    WHERE impressions_30d >= 100
      AND avg_position_30d IS NOT NULL
""").df()

print(f'Feature frame: {len(df):,} content items with >= 100 impressions in March 2026')
print(f'Columns: {list(df.columns)}')
print()

# --- Compute CTR and position tier (label computation, not features) ---
df['ctr_30d'] = df['clicks_30d'] / df['impressions_30d'] * 100

def assign_tier(pos):
    """Same tier boundaries as the starter dataset."""
    if pos <= 3:  return 'top_3'
    if pos <= 10: return 'page_1'
    if pos <= 20: return 'striking'
    if pos <= 50: return 'page_3_5'
    return 'deep'

df['position_tier'] = df['avg_position_30d'].apply(assign_tier)

# Tier median CTR
tier_median = df.groupby('position_tier')['ctr_30d'].median()
print('Median CTR by position tier (×100 percentages):')
print(tier_median.round(3).to_string())
print()

df['tier_median_ctr'] = df['position_tier'].map(tier_median)
df['is_under_ctr'] = (df['ctr_30d'] < df['tier_median_ctr']).astype(int)

print(f'Proxy label distribution:')
print(df['is_under_ctr'].value_counts().rename({0: 'at or above tier median', 1: 'below tier median'}))
print(f'Base rate: {df["is_under_ctr"].mean():.1%}')
print()

# Show a sample of the feature frame
FEATURES = ['avg_position_30d', 'impressions_30d', 'days_with_impressions',
            'engagement_rate_30d', 'position_stddev_30d']
show_cols = FEATURES + ['ctr_30d', 'position_tier', 'is_under_ctr']
print('Sample rows (5 features + label):')
print(df[show_cols].head(8).to_string(index=False))

Verified data contract query.


### Feature availability: "knowable at the decision moment because…"

| # | Feature | Knowable at the decision moment because… |
|---|---|---|
| 1 | `avg_position_30d` | Position is measured daily by Google Search Console before any editorial action is taken |
| 2 | `impressions_30d` | Impressions are passively observed search volume — no future data needed |
| 3 | `days_with_impressions` | Counts only past days with visibility; fully observable before the decision |
| 4 | `engagement_rate_30d` | GA4 engagement is measured independently of search clicks (filtered on `ga4_data_available IS TRUE` to exclude zero-filled rows) |
| 5 | `position_stddev_30d` | Position volatility is computed from historical daily position readings only |

---

### 3.5 — The Trap: deliberate leakage with `ctr_30d`

The label `is_under_ctr` is derived from `ctr_30d` (= clicks ÷ impressions). What happens if we add `ctr_30d` as a 6th feature? The model can reconstruct the label from the feature → precision@50 jumps toward perfect. Then we delete it and keep the honest number.

This is the leakage lesson from notebook 02, performed on real warehouse data.

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Prepare model data
model_data = df.copy()
# Fill NaN engagement (pages without GA4 data) with 0
model_data['engagement_rate_30d'] = model_data['engagement_rate_30d'].fillna(0)
# Fill NaN position_stddev (pages with only 1 day of data) with 0
model_data['position_stddev_30d'] = model_data['position_stddev_30d'].fillna(0)

y = model_data['is_under_ctr']

# ===== LEAKED MODEL: add ctr_30d as a feature =====
leaked_features = FEATURES + ['ctr_30d']
X_leaked = model_data[leaked_features]

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(
    X_leaked, y, test_size=0.25, random_state=42, stratify=y
)
model_leaked = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_leaked.fit(X_tr_l, y_tr_l)

proba_leaked = model_leaked.predict_proba(X_te_l)[:, 1]
top50_idx_leaked = proba_leaked.argsort()[::-1][:50]
p50_leaked = y_te_l.iloc[top50_idx_leaked].mean()

print('=' * 60)
print('LEAKED MODEL (ctr_30d included as a feature)')
print(f'  Features: {leaked_features}')
print(f'  Precision@50 = {p50_leaked:.3f}  ({int(p50_leaked * 50)} of 50 correct)')
print('=' * 60)
print()
print('⚠ The score is near-perfect because ctr_30d IS the quantity the')
print('  label is derived from. The model just learned: "if ctr_30d is')
print('  low relative to the position tier, predict 1." That is circular.')
print()

# ===== HONEST MODEL: remove ctr_30d =====
X_honest = model_data[FEATURES]

X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(
    X_honest, y, test_size=0.25, random_state=42, stratify=y
)
model_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_honest.fit(X_tr_h, y_tr_h)

proba_honest = model_honest.predict_proba(X_te_h)[:, 1]
top50_idx_honest = proba_honest.argsort()[::-1][:50]
p50_honest = y_te_h.iloc[top50_idx_honest].mean()

print('=' * 60)
print('HONEST MODEL (ctr_30d removed — the number we keep)')
print(f'  Features: {FEATURES}')
print(f'  Precision@50 = {p50_honest:.3f}  ({int(p50_honest * 50)} of 50 correct)')
print('=' * 60)
print()
print(f'Leakage lesson: adding ctr_30d inflated Precision@50 from')
print(f'  {p50_honest:.3f} → {p50_leaked:.3f}. The model\'s job is to predict')
print(f'  CTR opportunity *without seeing CTR itself*.')
print(f'  ctr_30d is DELETED — the honest Precision@50 = {p50_honest:.3f}.')

Verified data contract query.


## 4. Data limits

### Named limitation: Unbalanced panel — per-client history depth varies wildly

Not all clients entered the panel at the same time. Some have 17 months of GSC data; others have 3 months or less. This creates two problems:

1. **Feature availability differs by client.** A feature like "average position over the last 90 days" is meaningful for a client with 6+ months of history, but for a client whose data starts mid-February 2026, March 2026 is nearly their first full month — there is no baseline to compare against.

2. **GA4 coverage is partial.** Rows before a client's `ga4_data_start` have GA4 columns zero-filled (flagged `ga4_data_available = FALSE` or `NULL`). For these rows, engagement features are unavailable — not "zero engagement." A model that treats zeros as signal will learn the wrong pattern.

**Mitigation:** Always check `dim_clients.gsc_data_start` and `ga4_data_start` before defining time windows. Prefer per-client windows over one global calendar window. Filter engagement features on `ga4_data_available IS TRUE` (never `= TRUE` or `NOT FALSE` — those drop `NULL` rows silently).

In [21]:
# Evidence: how many clients have data starting in/before March 2026?
client_coverage = con.sql(f"""
    SELECT
        client_hash_id,
        gsc_data_start,
        ga4_data_start,
        CASE WHEN gsc_data_start <= DATE '2026-03-01' THEN 'has March GSC'
             ELSE 'no March GSC' END AS march_gsc_status
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('Client history coverage for March 2026:')
print(f"  Total clients in dim_clients:     {len(client_coverage)}")
has_march = (client_coverage['gsc_data_start'] <= pd.Timestamp('2026-03-01')).sum()
print(f"  Clients with GSC data by March:   {has_march}")
has_ga4 = (client_coverage['ga4_data_start'] <= pd.Timestamp('2026-03-01')).sum()
print(f"  Clients with GA4 data by March:   {has_ga4}")
print()

# Show the spread of start dates
print('GSC start date distribution:')
print(client_coverage['gsc_data_start'].describe())
print()
print('→ History depth varies widely: check per-client start dates before')
print('  defining any time window. A one-size-fits-all window will either')
print('  exclude short-history clients or include meaningless early rows.')

Verified data contract query.


## Self-check

Before you submit, confirm each line honestly:

- [x] Five plain-words contract answers (unit, table, window, label, excluded)
- [x] Three verification queries with outputs visible (grain, counts, availability with `IS TRUE`)
- [x] Five-feature frame with an "available when?" line per feature
- [x] Deliberate-leak experiment shown (ctr_30d) and removed — honest number kept
- [x] One named limitation of this data slice (unbalanced panel)
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.